# BRPMM Project - Process A & Process B Analysis

This notebook loads and analyzes the BRPMM (Batch Reaction Process Monitoring & Modelling) datasets for **Process A** and **Process B**.

## Overview
- **Process A**: Industrial process with 11 sensor features
- **Process B**: Fermentation/bioreactor process with 20 sensor features
- **Task**: Binary classification of batches as `accepted` or `rejected`

Each dataset contains:
- **Calibration data**: Labeled time-series data (with ground truth labels)
- **Test data**: Unlabeled time-series data (to predict on)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set styling
plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline

---
## 1. File Paths Configuration

In [2]:
BASE_PATH = Path('/home/renku/work/input-data/Project 2 - BRPMM')

# Process A paths
PROC_A_CALIB_DATA = BASE_PATH / 'process A' / 'calibration data' / 'calibration_data.csv'
PROC_A_CALIB_LABELS = BASE_PATH / 'process A' / 'calibration data' / 'calibration_labels.csv'
PROC_A_TEST_DATA = BASE_PATH / 'process A' / 'test data' / 'test_data.csv'

# Process B paths
PROC_B_CALIB_DATA = BASE_PATH / 'process B' / 'calibration data' / 'calibration_data.csv'
PROC_B_CALIB_LABELS = BASE_PATH / 'process B' / 'calibration data' / 'calibration_labels.csv'
PROC_B_TEST_DATA = BASE_PATH / 'process B' / 'test data' / 'test_data.csv'

# Submission template
TEMPLATE_PATH = BASE_PATH / 'BRPMM_template.csv'

print('All paths configured.')

All paths configured.


---
## 2. Process A — Data Loading

In [3]:
print('=' * 60)
print('PROCESS A - CALIBRATION DATA')
print('=' * 60)

# Load calibration data and labels
df_a_calib = pd.read_csv(PROC_A_CALIB_DATA)
df_a_calib_labels = pd.read_csv(PROC_A_CALIB_LABELS)

print(f'Calibration data shape: {df_a_calib.shape}')
print(f'Calibration labels shape: {df_a_calib_labels.shape}')
print(f'\nColumns: {list(df_a_calib.columns)}')
print(f'\nUnique batches in calib data: {df_a_calib["batch_id"].nunique()}')
print(f'Unique batches in calib labels: {df_a_calib_labels["batch_id"].nunique()}')
print(f'\nLabel distribution:')
print(df_a_calib_labels['label'].value_counts())

df_a_calib.head(10)

PROCESS A - CALIBRATION DATA
Calibration data shape: (415165, 12)
Calibration labels shape: (161, 2)

Columns: ['batch_id', 'flow_rate_1', 'density', 'flow_rate_2', 'pressure_1', 'temperature_1', 'temperature_2', 'temperature_3', 'temperature_4', 'pressure_2', 'pressure_3', 'pressure_4']

Unique batches in calib data: 161
Unique batches in calib labels: 161

Label distribution:
label
accepted    121
rejected     40
Name: count, dtype: int64


,batch_id,flow_rate_1,density,flow_rate_2,pressure_1,temperature_1,temperature_2,temperature_3,temperature_4,pressure_2,pressure_3,pressure_4
0,A_003,132.220502,1.377930,1850.478922,26.497453,13.977803,84.619420,25.302022,-12.250281,9.951612,65.233459,6.255296
1,A_003,131.312045,1.377903,1859.290763,26.072998,13.987786,84.616369,25.833591,-12.253354,8.583511,64.113327,2.855173
2,A_003,130.968035,1.378086,1853.816580,25.881007,13.991541,84.590239,26.186738,-12.252625,8.269868,62.672264,2.970329
3,A_003,131.276685,1.378135,1857.069601,25.745995,14.003087,84.627614,26.440939,-12.258613,9.562431,64.997071,2.471810
4,A_003,131.098303,1.378335,1850.890328,25.706881,13.993630,84.607380,26.636520,-12.273808,8.037131,62.756263,2.266525
5,A_003,131.485017,1.378298,1847.252357,25.697489,13.990058,84.587446,26.603443,-12.272173,6.472858,61.781781,1.745423
6,A_003,131.204387,1.378554,1853.240541,25.671890,13.976801,84.532838,26.545632,-12.300432,9.192507,63.534785,0.060839
7,A_003,131.279589,1.378553,1851.031784,25.634765,13.996773,84.471405,26.524824,-12.312557,12.183916,67.663547,8.082502
8,A_003,131.514473,1.378558,1856.490862,25.716925,13.984863,84.405819,26.501945,-12.313187,10.610563,66.234667,7.028169
9,A_003,131.143152,1.378517,1863.049572,25.669931,13.959253,84.309003,26.459929,-12.309555,10.711081,66.719699,6.647906


In [4]:
print('=' * 60)
print('PROCESS A - TEST DATA')
print('=' * 60)

# Load test data
df_a_test = pd.read_csv(PROC_A_TEST_DATA)

print(f'Test data shape: {df_a_test.shape}')
print(f'Unique batches in test data: {df_a_test["batch_id"].nunique()}')

df_a_test.head(10)

PROCESS A - TEST DATA
Test data shape: (414301, 12)
Unique batches in test data: 161


,batch_id,flow_rate_1,density,flow_rate_2,pressure_1,temperature_1,temperature_2,temperature_3,temperature_4,pressure_2,pressure_3,pressure_4
0,A_001,130.593684,1.376252,1858.910021,24.965531,14.139781,91.395977,26.390206,-12.362755,10.717542,66.996042,4.123406
1,A_001,130.947459,1.376273,1852.088462,25.092851,14.156456,91.342414,26.381275,-12.378745,10.896834,68.650397,6.649504
2,A_001,131.558278,1.376291,1852.938488,25.225541,14.182462,91.130165,26.344407,-12.390484,10.603608,67.523670,5.703887
3,A_001,132.231594,1.376423,1854.849648,25.172916,14.160006,90.983129,26.199301,-12.404257,8.996890,66.110962,5.185230
4,A_001,132.215212,1.376568,1850.840999,25.048504,14.199144,90.781370,26.101028,-12.403928,7.519596,64.508000,4.071038
5,A_001,131.391118,1.376575,1829.644078,25.119752,14.212308,90.585797,26.540520,-12.380874,7.695345,65.772826,4.756819
6,A_001,131.098033,1.376659,1846.736186,25.117692,14.188861,90.383745,26.534579,-12.357619,4.456480,60.800934,0.211679
7,A_001,131.160178,1.376718,1838.084206,24.957899,14.219194,90.201097,26.440544,-12.338932,6.298945,62.462747,1.781227
8,A_001,130.697362,1.376771,1824.609638,24.910850,14.227488,90.016473,26.531605,-12.326394,8.693282,66.005196,3.730357
9,A_001,130.886349,1.376941,1853.589008,25.066372,14.233160,89.843422,26.457717,-12.312555,5.565863,62.452855,0.664248


---
## 3. Process B — Data Loading

In [5]:
print('=' * 60)
print('PROCESS B - CALIBRATION DATA')
print('=' * 60)

# Load calibration data and labels
df_b_calib = pd.read_csv(PROC_B_CALIB_DATA)
df_b_calib_labels = pd.read_csv(PROC_B_CALIB_LABELS)

print(f'Calibration data shape: {df_b_calib.shape}')
print(f'Calibration labels shape: {df_b_calib_labels.shape}')
print(f'\nColumns: {list(df_b_calib.columns)}')
print(f'\nUnique batches in calib data: {df_b_calib["batch_id"].nunique()}')
print(f'Unique batches in calib labels: {df_b_calib_labels["batch_id"].nunique()}')
print(f'\nLabel distribution:')
print(df_b_calib_labels['label'].value_counts())

df_b_calib.head(10)

PROCESS B - CALIBRATION DATA
Calibration data shape: (200040, 21)
Calibration labels shape: (40, 2)

Columns: ['batch_id', 'substrate_concentration', 'dissolved_oxygen', 'biomass_concentration', 'penicillin_concentration', 'fermentation_volume', 'dissolved_co2', 'ph', 'reactor_temperature', 'reaction_heat', 'feed_rate', 'aeration_rate', 'agitator_power', 'feed_temperature', 'water_flow_rate', 'hot_cold_switch', 'base_flow', 'acid_flow', 'feed_substate_concentration', 'cooling_water_temperature', 'heating_water_temperature']

Unique batches in calib data: 40
Unique batches in calib labels: 40

Label distribution:
label
accepted    35
rejected     5
Name: count, dtype: int64


,batch_id,substrate_concentration,dissolved_oxygen,biomass_concentration,penicillin_concentration,fermentation_volume,dissolved_co2,ph,reactor_temperature,reaction_heat,...,aeration_rate,agitator_power,feed_temperature,water_flow_rate,hot_cold_switch,base_flow,acid_flow,feed_substate_concentration,cooling_water_temperature,heating_water_temperature
0,B_001,17.137256,1.158435,0.149299,0.000000e+00,108.291986,0.311289,5.000000,298.000000,0.000000,...,7.028857,30.000000,296.000000,0.000000,0.0,0.000000e+00,0.0,600.0,290.0,323.0
1,B_001,17.136622,1.155105,0.149604,8.666972e-08,108.291446,0.454900,4.999867,298.000000,1.982014,...,7.028886,30.000007,295.999971,0.000039,0.0,1.593963e-07,0.0,600.0,290.0,323.0
2,B_001,17.135986,1.152940,0.149911,1.734518e-07,108.290907,0.347303,4.999738,298.000000,3.968017,...,7.028916,30.000014,295.999941,0.000137,0.0,3.118717e-07,0.0,600.0,290.0,323.0
3,B_001,17.135349,1.156060,0.150218,2.603460e-07,108.290368,0.483830,4.999612,298.000001,5.957998,...,7.028946,30.000019,295.999910,0.000277,0.0,4.855826e-07,0.0,600.0,290.0,323.0
4,B_001,17.134710,1.154090,0.150525,3.473527e-07,108.289828,0.361944,4.999491,298.000001,7.951968,...,7.028974,30.000020,295.999881,0.000458,0.0,6.792308e-07,0.0,600.0,290.0,323.0
5,B_001,17.134070,1.156874,0.150834,4.344720e-07,108.289289,0.463542,4.999373,298.000001,9.949937,...,7.029003,30.000023,295.999852,0.000665,0.0,8.916012e-07,0.0,600.0,290.0,323.0
6,B_001,17.133428,1.157518,0.151143,5.217043e-07,108.288750,0.487528,4.999261,298.000002,11.951916,...,7.029031,30.000030,295.999823,0.000883,0.0,1.121417e-06,0.0,600.0,290.0,323.0
7,B_001,17.132785,1.154504,0.151452,6.090499e-07,108.288210,0.469761,4.999153,298.000002,13.957915,...,7.029058,30.000039,295.999796,0.001098,0.0,1.367348e-06,0.0,600.0,290.0,323.0
8,B_001,17.132140,1.156656,0.151762,6.965090e-07,108.287671,0.453103,4.999052,298.000002,15.967945,...,7.029086,30.000048,295.999769,0.001296,0.0,1.628017e-06,0.0,600.0,290.0,323.0
9,B_001,17.131494,1.156391,0.152073,7.840821e-07,108.287132,0.391226,4.998956,298.000001,17.982018,...,7.029113,30.000057,295.999744,0.001471,0.0,1.902006e-06,0.0,600.0,290.0,323.0


In [ ]:
print('=' * 60)
print('PROCESS B - TEST DATA')
print('=' * 60)

# Load test data
df_b_test = pd.read_csv(PROC_B_TEST_DATA)

print(f'Test data shape: {df_b_test.shape}')
print(f'Unique batches in test data: {df_b_test["batch_id"].nunique()}')

df_b_test.head(10)

PROCESS B - TEST DATA


---
## 4. Time-Series Statistics per Batch

We compute summary statistics (mean, std, min, max, etc.) for each batch to
convert the variable-length time series into fixed-length feature vectors
suitable for classical ML models.

In [ ]:
def compute_batch_features(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    Aggregate time-series data into per-batch features.
    
    For each batch and each feature column, compute:
    - mean, std, min, max, first, last
    - slope (linear trend over time)
    
    Returns a DataFrame with one row per batch.
    """
    agg_funcs = ['mean', 'std', 'min', 'max', 'first', 'last']
    
    # Group by batch_id and aggregate
    grouped = df.groupby('batch_id')[feature_cols].agg(agg_funcs)
    
    # Flatten MultiIndex columns: e.g., ('flow_rate_1', 'mean') -> 'flow_rate_1__mean'
    grouped.columns = ['__'.join(col).strip() for col in grouped.columns.values]
    
    return grouped.reset_index()


print('Feature engineering function defined.')

### 4.1 Process A — Feature Engineering

In [ ]:
# Determine feature columns (all except batch_id)
feature_cols_a = [col for col in df_a_calib.columns if col != 'batch_id']
print(f'Process A feature columns ({len(feature_cols_a)}): {feature_cols_a}')

# Compute batch-level features for calibration data
df_a_features = compute_batch_features(df_a_calib, feature_cols_a)

# Merge with labels
df_a_train = df_a_features.merge(df_a_calib_labels, on='batch_id', how='inner')

print(f'\nTraining features shape: {df_a_train.shape}')
print(f'Label distribution in training set:')
print(df_a_train['label'].value_counts())

df_a_train.head()

### 4.2 Process B — Feature Engineering

In [ ]:
# Determine feature columns (all except batch_id)
feature_cols_b = [col for col in df_b_calib.columns if col != 'batch_id']
print(f'Process B feature columns ({len(feature_cols_b)}): {feature_cols_b}')

# Compute batch-level features for calibration data
df_b_features = compute_batch_features(df_b_calib, feature_cols_b)

# Merge with labels
df_b_train = df_b_features.merge(df_b_calib_labels, on='batch_id', how='inner')

print(f'\nTraining features shape: {df_b_train.shape}')
print(f'Label distribution in training set:')
print(df_b_train['label'].value_counts())

df_b_train.head()

---
## 5. Exploratory Data Analysis — Process A

In [ ]:
# Plot sample time series for accepted vs rejected batches
def plot_sample_batches(df, labels_df, feature_cols, title_prefix, n_samples=3):
    accepted_ids = labels_df[labels_df['label'] == 'accepted']['batch_id'].values[:n_samples]
    rejected_ids = labels_df[labels_df['label'] == 'rejected']['batch_id'].values[:n_samples]
    
    n_features = len(feature_cols)
    fig, axes = plt.subplots(n_features, 2, figsize=(14, 3 * n_features))
    
    for i, col in enumerate(feature_cols):
        # Accepted batches
        ax_acc = axes[i, 0] if n_features > 1 else axes[0]
        for bid in accepted_ids:
            batch_data = df[df['batch_id'] == bid]
            ax_acc.plot(batch_data[col].values, alpha=0.7, linewidth=0.5, label=bid)
        ax_acc.set_title(f'{col} - Accepted Batches')
        ax_acc.set_xlabel('Time Step')
        ax_acc.legend(fontsize=7, loc='best')
        
        # Rejected batches
        ax_rej = axes[i, 1] if n_features > 1 else axes[1]
        for bid in rejected_ids:
            batch_data = df[df['batch_id'] == bid]
            ax_rej.plot(batch_data[col].values, alpha=0.7, linewidth=0.5, label=bid)
        ax_rej.set_title(f'{col} - Rejected Batches')
        ax_rej.set_xlabel('Time Step')
        ax_rej.legend(fontsize=7, loc='best')
    
    plt.suptitle(f'{title_prefix}: Time Series Comparison (Accepted vs Rejected)', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_sample_batches(df_a_calib, df_a_calib_labels, feature_cols_a, 'Process A', n_samples=3)

In [ ]:
# Boxplot of aggregated features by label
agg_cols = [c for c in df_a_train.columns if c.endswith('__mean')]
n_cols = len(agg_cols)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(agg_cols):
    if i < len(axes):
        for label in ['accepted', 'rejected']:
            subset = df_a_train[df_a_train['label'] == label][col]
            axes[i].hist(subset, alpha=0.5, label=label, bins=20)
        axes[i].set_title(col.replace('__mean', ''), fontsize=9)
        axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Process A: Distribution of Mean Values by Label', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Exploratory Data Analysis — Process B

In [ ]:
# Select a subset of key features for Process B (too many to plot all at once)
key_features_b = ['substrate_concentration', 'dissolved_oxygen', 'biomass_concentration',
                   'penicillin_concentration', 'fermentation_volume', 'ph',
                   'reactor_temperature', 'aeration_rate', 'agitator_power']

plot_sample_batches(df_b_calib, df_b_calib_labels, key_features_b, 'Process B', n_samples=3)

In [ ]:
# Boxplot of aggregated features by label for Process B
agg_cols_b = [c for c in df_b_train.columns if c.endswith('__mean')]
n_cols_b = len(agg_cols_b)

fig, axes = plt.subplots(5, 4, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(agg_cols_b):
    if i < len(axes):
        for label in ['accepted', 'rejected']:
            subset = df_b_train[df_b_train['label'] == label][col]
            axes[i].hist(subset, alpha=0.5, label=label, bins=20)
        axes[i].set_title(col.replace('__mean', ''), fontsize=8)
        axes[i].legend(fontsize=6)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Process B: Distribution of Mean Values by Label', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Correlation Analysis

In [ ]:
# Process A correlation heatmap (mean features)
feature_mean_cols_a = [c for c in df_a_train.columns if c.endswith('__mean')]
corr_a = df_a_train[feature_mean_cols_a].corr()

# Rename columns for readability
rename_a = {c: c.replace('__mean', '') for c in feature_mean_cols_a}
corr_a = corr_a.rename(index=rename_a, columns=rename_a)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_a, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Process A: Feature Correlation Matrix (Mean Values)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Process B correlation heatmap (mean features)
feature_mean_cols_b = [c for c in df_b_train.columns if c.endswith('__mean')]
corr_b = df_b_train[feature_mean_cols_b].corr()

# Rename columns for readability
rename_b = {c: c.replace('__mean', '') for c in feature_mean_cols_b}
corr_b = corr_b.rename(index=rename_b, columns=rename_b)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_b, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Process B: Feature Correlation Matrix (Mean Values)', fontsize=14)
plt.tight_layout()
plt.show()